# Comptage des codes post-coordonnés CIM-11

**Objectif unique** : compter le nombre de couples uniques (code de base, code d'extension)
réellement autorisés par l'API OMS, tous axes confondus — au total et par chapitre CIM-11.

Aucune autre information n'est conservée que ce compte final, pour limiter l'usage mémoire.

In [ ]:
import requests          # pour appeler l'API REST de l'OMS
import json              # pas utilisé directement ici, mais utile si on inspecte des réponses brutes
import time              # pour gérer les pauses (anti-erreur 429, attente entre tentatives)
import pandas as pd      # pour manipuler les tableaux de données (codes, résultats, comptages)
from threading import Lock                                  # pour protéger le token partagé entre threads
from concurrent.futures import ThreadPoolExecutor, as_completed  # pour paralléliser les appels API

## Configuration de l'API OMS (ICD-11)

On définit ici les identifiants de connexion et les paramètres globaux :
- `CLIENT_ID` / `CLIENT_SECRET` : identifiants OAuth2 fournis par l'OMS pour accéder à l'API.
- `TOKEN_URL` : adresse où on échange ces identifiants contre un jeton d'accès temporaire.
- `MMS_ROOT` : adresse racine de la classification CIM-11 (version "MMS" = linéarisation utilisée pour le codage médical).
- `MAX_WORKERS` : nombre de requêtes envoyées en parallèle (pour aller plus vite sans surcharger l'API).
- `LANGUE` : on demande les réponses en français.
- `_token`, `_token_exp`, `_lock` : variables internes pour stocker le jeton actif et éviter que plusieurs threads le renouvellent en même temps.

In [1]:
from google.colab import userdata

# Identifiants OAuth2 — récupérés depuis les secrets Colab (icône clé 🔑 à gauche)
try:
    CLIENT_ID     = userdata.get('WHO_CLIENT_ID')
    CLIENT_SECRET = userdata.get('WHO_CLIENT_SECRET')
except Exception:
    CLIENT_ID = CLIENT_SECRET = None

if not CLIENT_ID or not CLIENT_SECRET:
    raise ValueError(
        "Identifiants OMS manquants.\n"
        "1. Crée un compte sur https://icd.who.int/icdapi pour obtenir un client_id et un client_secret (gratuit)\n"
        "2. Dans Colab, ouvre le panneau Secrets (icône clé 🔑 à gauche)\n"
        "3. Ajoute deux secrets : WHO_CLIENT_ID et WHO_CLIENT_SECRET\n"
        "4. Active l'accès à ce notebook pour les deux secrets, puis relance cette cellule"
    )

# Adresse pour obtenir un jeton d'accès
TOKEN_URL = "https://icdaccessmanagement.who.int/connect/token"

# Adresse racine de la CIM-11 (linéarisation MMS, version 2024-01)
MMS_ROOT = "https://id.who.int/icd/release/11/2024-01/mms"

# Nombre de requêtes en parallèle (10 = bon compromis vitesse / respect de l'API)
MAX_WORKERS = 10

# Langue des réponses de l'API
LANGUE = 'fr'

# Variables internes pour la gestion du jeton (ne pas modifier à la main)
_token     = None   # contiendra le jeton actif une fois obtenu
_token_exp = 0       # timestamp d'expiration du jeton
_lock      = Lock()  # verrou pour éviter que 2 threads renouvellent le jeton en même temps

ValueError: Identifiants OMS manquants.
1. Crée un compte sur https://icd.who.int/icdapi pour obtenir un client_id et un client_secret (gratuit)
2. Dans Colab, ouvre le panneau Secrets (icône clé 🔑 à gauche)
3. Ajoute deux secrets : WHO_CLIENT_ID et WHO_CLIENT_SECRET
4. Active l'accès à ce notebook pour les deux secrets, puis relance cette cellule

## Authentification OAuth2 et requêtes HTTP sécurisées

Deux fonctions :

- `get_token()` : demande un nouveau jeton d'accès si l'ancien a expiré (ou n'existe pas encore).
  Le jeton OMS est valide environ 60 minutes ; on le garde en mémoire pour ne pas le redemander à chaque appel.

- `hdrs()` : construit les en-têtes HTTP à envoyer avec chaque requête (jeton, langue, version de l'API).

- `get_url()` : fait la requête vers l'API, avec gestion automatique des erreurs courantes :
  - **429** (trop de requêtes) → on attend de plus en plus longtemps avant de réessayer.
  - **401** (jeton expiré) → on force le renouvellement du jeton puis on réessaie.
  - Si ça échoue après plusieurs tentatives, on renvoie `None` plutôt que de planter tout le notebook.

In [ ]:
def get_token() -> str:
    """Renvoie un jeton OAuth2 valide. Le renouvelle automatiquement si besoin."""
    global _token, _token_exp
    with _lock:  # un seul thread à la fois peut renouveler le jeton
        # Si le jeton actuel est encore valide (avec 30s de marge de sécurité), on le réutilise
        if time.time() < _token_exp - 30:
            return _token

        print('Renouvellement du token...')
        r = requests.post(TOKEN_URL, data={
            'client_id':     CLIENT_ID,
            'client_secret': CLIENT_SECRET,
            'scope':         'icdapi_access',
            'grant_type':    'client_credentials',
        }, timeout=15)
        r.raise_for_status()  # lève une erreur si la demande de jeton échoue

        d          = r.json()
        _token     = d['access_token']
        _token_exp = time.time() + d.get('expires_in', 3600)  # date d'expiration du nouveau jeton
        print(f'Token OK — valide {d.get("expires_in", 3600) // 60} minutes')
        return _token


def hdrs() -> dict:
    """Construit les en-têtes HTTP à joindre à chaque requête vers l'API."""
    return {
        'Authorization':   f'Bearer {get_token()}',  # jeton d'accès
        'Accept':          'application/json',        # on veut du JSON en réponse
        'Accept-Language': LANGUE,                    # réponses en français
        'API-Version':     'v2',                       # version de l'API OMS utilisée
    }


def get_url(url: str, retries: int = 5):
    """
    Appelle l'API à l'adresse `url` et renvoie la réponse JSON.
    Réessaie automatiquement en cas d'erreur temporaire (429, 401, timeout...).
    Renvoie None si toutes les tentatives échouent.
    """
    for tentative in range(retries):
        try:
            r = requests.get(url, headers=hdrs(), timeout=25)

            if r.status_code == 429:
                # Trop de requêtes envoyées : on attend de plus en plus longtemps (1s, 2s, 4s, 8s...)
                time.sleep(2 ** tentative)
                continue

            if r.status_code == 401:
                # Jeton expiré : on force son renouvellement au prochain appel
                global _token_exp
                _token_exp = 0
                time.sleep(1)
                continue

            r.raise_for_status()  # lève une erreur pour les autres codes HTTP anormaux
            return r.json()

        except Exception:
            # Erreur réseau ou autre : on réessaie, sauf si c'était la dernière tentative
            if tentative == retries - 1:
                return None
            time.sleep(1 + tentative)

    return None

In [ ]:
# Test : on vérifie que l'authentification fonctionne et que la racine CIM-11 répond
get_token()
racine = get_url(MMS_ROOT)
print(f"Chapitres trouvés à la racine : {len(racine.get('child', []))}")

Renouvellement du token...
Token OK — valide 60 minutes
Chapitres trouvés à la racine : 28


## Chargement des codes de base

On charge le fichier `cim11_termes.csv`, qui contient tous les termes CIM-11
(codes officiels + synonymes/index terms).

On ne garde que les lignes de type `title` : ce sont les codes "officiels"
(un code = une ligne), avec leur URI d'API associé. C'est notre liste de
départ pour le comptage : **34 663 codes de base**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # connecte le notebook à ton Google Drive

# Chemin vers le fichier des termes CIM-11
PATH_SYN = "/content/drive/MyDrive/Colab_Notebooks/Serenic_M/api/cim11_termes.csv"

# Chargement du fichier (encoding utf-8-sig car le CSV contient un BOM)
df_syn = pd.read_csv(PATH_SYN, encoding='utf-8-sig')

print(f"Nombre total de lignes (tous types confondus) : {len(df_syn)}")
df_syn.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Nombre total de lignes (tous types confondus) : 119521


,code,uri,texte,type
0,4A20,http://id.who.int/icd/release/11/2024-01/mms/6...,Déficits immunitaires acquis,title
1,4B40,http://id.who.int/icd/release/11/2024-01/mms/6...,Maladies du thymus,title
2,4B4Z,http://id.who.int/icd/release/11/2024-01/mms/1...,"Maladies du système immunitaire, sans précision",title
3,4B4Z,http://id.who.int/icd/release/11/2024-01/mms/1...,maladie immunitaire SAI,indexTerm
4,4B4Z,http://id.who.int/icd/release/11/2024-01/mms/1...,déficit immunitaire SAI,indexTerm


## Extraction des codes de base

`df_syn` contient plusieurs types de lignes : des codes officiels (`title`)
et des synonymes/termes d'index (`indexTerm`, etc.). Pour notre comptage, on
ne veut qu'**un code de base par ligne**, donc on filtre sur `type == 'title'`.

On ne garde que les colonnes `code` (le code CIM-11) et `uri` (l'adresse API
qui permet d'interroger ce code), et on supprime les doublons éventuels.

In [ ]:
# On garde uniquement les codes "officiels" (type == 'title'), avec leur code et leur URI
df_codes = df_syn[df_syn['type'] == 'title'][['code', 'uri']].drop_duplicates().reset_index(drop=True)

print(f"Codes de base chargés : {len(df_codes)}")
df_codes.head()

Codes de base chargés : 34663


,code,uri
0,4A20,http://id.who.int/icd/release/11/2024-01/mms/6...
1,4B40,http://id.who.int/icd/release/11/2024-01/mms/6...
2,4B4Z,http://id.who.int/icd/release/11/2024-01/mms/1...
3,4B4Y,http://id.who.int/icd/release/11/2024-01/mms/1...
4,3C0Y,http://id.who.int/icd/release/11/2024-01/mms/1...


In [ ]:
# Test rapide : on vérifie qu'on a bien récupéré le nombre attendu de codes
assert len(df_codes) > 0
print(f"OK — {len(df_codes)} codes de base prêts pour la suite.")

OK — 34663 codes de base prêts pour la suite.


## Récupération des 28 chapitres CIM-11

La racine `MMS_ROOT` a 28 enfants directs : ce sont les 28 chapitres de la CIM-11
(maladies infectieuses, tumeurs, etc., plus les chapitres spéciaux comme l'extension X).

On récupère leur URI et leur titre une seule fois, et on les stocke dans un
dictionnaire `CHAPITRE_ROOTS` (URI → nom du chapitre). C'est notre point de
repère pour savoir, plus tard, quand on a atteint le sommet de la hiérarchie.

In [ ]:
# Récupère les enfants directs de la racine = les 28 chapitres de la CIM-11
racine = get_url(MMS_ROOT)
chapitre_uris = racine.get('child', [])

CHAPITRE_ROOTS = {}  # URI (en https) -> nom lisible du chapitre

for i, uri in enumerate(chapitre_uris, start=1):
    uri_https = uri.replace('http://', 'https://')  # l'API renvoie du http, on uniformise en https
    data = get_url(uri_https)
    titre = data.get('title', {}).get('@value', f'Chapitre {i}') if data else f'Chapitre {i}'
    CHAPITRE_ROOTS[uri_https] = f"{i:02d} - {titre}"

print(f"Chapitres récupérés : {len(CHAPITRE_ROOTS)}")
for nom in CHAPITRE_ROOTS.values():
    print(nom)

Chapitres récupérés : 28
01 - Certaines maladies infectieuses ou parasitaires
02 - Tumeurs
03 - Maladies du sang ou des organes hématopoïétiques
04 - Maladies du système immunitaire
05 - Maladies endocriniennes, nutritionnelles ou métaboliques
06 - Troubles mentaux, comportementaux ou neurodéveloppementaux
07 - Troubles du cycle veille-sommeil
08 - Maladies du système nerveux
09 - Maladies de l'appareil visuel
10 - Maladies de l'oreille ou de l'apophyse mastoïde
11 - Maladies de l'appareil circulatoire
12 - Maladies de l'appareil respiratoire
13 - Maladies de l'appareil digestif
14 - Maladies de la peau
15 - Maladies du système musculosquelettique ou du tissu conjonctif
16 - Maladies de l'appareil génito-urinaire
17 - Affections liées à la santé sexuelle
18 - Grossesse, accouchement ou puerpéralité
19 - Certaines affections dont l'origine se situe dans la période périnatale
20 - Anomalies du développement
21 - Symptômes, signes ou résultats d'examen clinique, non classés ailleurs
22 - 

## Retrouver le chapitre d'un code à partir de son URI

`cim11_termes.csv` ne donne pas directement le chapitre d'un code. On le
retrouve en remontant la hiérarchie CIM-11 (`parent`) depuis le code jusqu'à
tomber sur l'un des 28 chapitres racines.

Pour ne pas refaire ce travail à chaque fois (beaucoup de codes partagent les
mêmes ancêtres), on garde un **cache global** (`chapitre_cache`) : chaque URI
déjà résolue n'est plus jamais recalculée, ce qui limite fortement le nombre
d'appels API au fil de l'avancement.

In [ ]:
# Cache global : associe chaque URI déjà résolue à son chapitre.
# Partagé par tous les codes traités -> évite de refaire le même calcul.
chapitre_cache = {}

def trouver_chapitre(uri: str, profondeur_max: int = 15) -> str:
    """
    Remonte la hiérarchie CIM-11 depuis `uri` jusqu'au chapitre racine.
    Utilise CHAPITRE_ROOTS (sommets) et chapitre_cache (mémoire des calculs déjà faits).
    Renvoie le nom du chapitre, ou None si aucun parent n'est trouvé (cas rare).
    """
    uri = uri.replace('http://', 'https://')

    # Cas direct : c'est déjà un chapitre racine
    if uri in CHAPITRE_ROOTS:
        return CHAPITRE_ROOTS[uri]

    # Cas déjà connu : on a déjà résolu cette URI avant
    if uri in chapitre_cache:
        return chapitre_cache[uri]

    chemin = []      # URIs traversées dans cet appel (pour les mettre en cache à la fin)
    courant = uri

    for _ in range(profondeur_max):
        chemin.append(courant)

        if courant in CHAPITRE_ROOTS:
            resultat = CHAPITRE_ROOTS[courant]
            for u in chemin:
                chapitre_cache[u] = resultat  # on mémorise tout le chemin parcouru
            return resultat

        if courant in chapitre_cache:
            resultat = chapitre_cache[courant]
            for u in chemin:
                chapitre_cache[u] = resultat
            return resultat

        data = get_url(courant)
        if not data or not data.get('parent'):
            return None  # orphelin : pas de parent trouvé, cas très rare

        # On prend le premier parent (suffisant pour situer le chapitre)
        courant = data['parent'][0].replace('http://', 'https://')

    return None  # profondeur max atteinte sans trouver de chapitre

In [ ]:
# Test sur 5 codes de base pris au hasard
for code_test, uri_test in df_codes.sample(5, random_state=42).values:
    chap = trouver_chapitre(uri_test)
    print(f"{code_test} -> {chap}")

DB33.4Z -> 13 - Maladies de l'appareil digestif
PL13.50 -> 23 - Causes externes de morbidité ou de mortalité
XM6DX1 -> 28 - Codes d'extension
LD24.4 -> 20 - Anomalies du développement
8A01.2 -> 08 - Maladies du système nerveux


## Récupération des axes de post-coordination d'un code

Pour un code de base donné, l'API renvoie une liste d'**axes** (ex. `histopathology`,
`associatedWith`, `temporalPatterns`...). Chaque axe a :
- un nom (`axe_nom`)
- une règle `allow_multiple` (`NotAllowed` = un seul code autorisé sur cet axe,
  `AllowAlways` = plusieurs codes possibles en même temps)
- une liste d'URI (`scale_entities`) = les valeurs autorisées sur cet axe

À ce stade, ce sont encore des **URI**, pas des codes CIM-11 lisibles.

In [ ]:
def get_postcoord(uri: str) -> list:
    """
    Récupère la liste des axes de post-coordination autorisés pour un code,
    à partir de son URI. Renvoie une liste de dictionnaires (un par axe).
    Renvoie une liste vide si le code n'a pas d'axe (= pas post-coordonnable).
    """
    data = get_url(uri)
    if not data:
        return []

    axes = []
    for axe in data.get('postcoordinationScale', []):
        axe_nom        = axe.get('axisName', '').split('/')[-1]
        allow_multiple = axe.get('allowMultipleValues', 'NotAllowed')
        scale_entities = axe.get('scaleEntity', [])

        # On exclut toutes les variantes "alternative" d'un axe principal
        # (ex: hasAlternativeSeverity1, hasAlternativeSeverity2, etc.)
        # Raison : un cas clinique n'utilise qu'un seul axe de sévérité à la fois,
        # garder les alternatives multiplierait artificiellement les combinaisons.
        if axe_nom.startswith('hasAlternative'):
            continue

        axes.append({
            'axe_nom':        axe_nom,
            'allow_multiple': allow_multiple,
            'scale_entities': [e.replace('http://', 'https://') for e in scale_entities],
        })

    return axes


## Transformer les URI d'un axe en vrais codes CIM-11

Les `scale_entities` d'un axe sont parfois des **catégories** (pas directement
un code final) qui ont elles-mêmes des enfants (`child`). Il faut descendre
récursivement jusqu'à trouver tous les codes réels.

**Optimisation mémoire/vitesse** : beaucoup d'axes différents (sur des codes
de base différents) pointent vers les **mêmes** listes d'URI (ex. les axes
"sévérité" ou "temporalité" sont souvent identiques d'un code à l'autre).
On garde donc un petit cache (`axe_cache`) qui associe une liste d'URI déjà
résolue à son résultat, pour ne jamais refaire le même travail deux fois.

### Garde-fous contre les axes trop volumineux

Certains axes CIM-11 (notamment `associatedWith` qui peut référencer toutes les maladies, ou `histopathology` avec ses milliers de types histologiques) sont **trop volumineux** pour être résolus en temps raisonnable.

On définit deux limites :
- `MAX_VALEURS_AXE = 5000` : au-delà, on considère l'axe inutilisable
- `MAX_TEMPS_AXE = 300` (5 min) : au-delà, on abandonne

Quand une de ces limites est atteinte, `get_valeurs_axe` lève une exception `AxeTropGrosError`. L'exception remonte jusqu'à la boucle principale qui marque le code racine concerné comme **"impossible à traiter"** (stocké dans `codes_impossibles`).

Ces codes seront exportés dans un fichier à part à la fin du traitement.


In [ ]:
# Garde-fous pour éviter de bloquer sur des axes trop volumineux
MAX_VALEURS_AXE = 5000   # Si un axe a plus de 5000 valeurs, on l'abandonne
MAX_TEMPS_AXE   = 300    # Si la résolution d'un axe dure plus de 5 min, on l'abandonne


class AxeTropGrosError(Exception):
    """
    Levée quand la résolution d'un axe dépasse les limites fixées
    (nombre de valeurs ou temps de calcul). Le code racine entier est
    alors considéré comme "impossible à traiter".
    """
    pass


In [ ]:
# Cache global : associe une liste d'URI de départ (clé figée, triée) à la liste de codes déjà résolue.
# Évite de redescendre dans la hiérarchie pour des axes identiques partagés par plusieurs codes.
axe_cache = {}

def get_valeurs_axe(scale_entities: list) -> list:
    """
    Pour une liste d'URI de départ d'un axe, descend récursivement dans les
    enfants jusqu'à trouver tous les codes CIM-11 réels.
    Renvoie une liste de codes (sans doublon).

    Lève AxeTropGrosError si :
    - l'axe dépasse MAX_VALEURS_AXE valeurs (axe énorme comme associatedWith)
    - la résolution dure plus de MAX_TEMPS_AXE secondes (axe lent à explorer)
    Dans ces cas, le code racine sera marqué comme "impossible à traiter".
    """
    cle_cache = tuple(sorted(scale_entities))  # clé stable pour le cache
    if cle_cache in axe_cache:
        return axe_cache[cle_cache]

    codes = []
    vus = set()      # URI déjà explorées dans cet appel, pour éviter les boucles infinies
    t0 = time.time()  # début de l'exploration, pour le timeout

    def explorer(uri):
        # Garde-fous : on abandonne l'axe s'il est trop gros ou trop long à résoudre
        if len(codes) > MAX_VALEURS_AXE:
            raise AxeTropGrosError(
                f"Axe trop volumineux : >{MAX_VALEURS_AXE} valeurs résolues"
            )
        if time.time() - t0 > MAX_TEMPS_AXE:
            raise AxeTropGrosError(
                f"Axe trop long à résoudre : >{MAX_TEMPS_AXE}s écoulés "
                f"({len(codes)} valeurs récupérées avant abandon)"
            )

        if uri in vus:
            return
        vus.add(uri)

        data = get_url(uri)
        if not data:
            return

        code = data.get('code')
        if code:
            codes.append(code)

        # Même sans code direct, on continue d'explorer les enfants (catégories intermédiaires)
        for child_uri in data.get('child', []):
            explorer(child_uri.replace('http://', 'https://'))

    for uri in scale_entities:
        explorer(uri)

    axe_cache[cle_cache] = codes  # on mémorise le résultat pour les prochains axes identiques
    return codes


In [ ]:
# Change juste cette ligne pour tester un autre code
CODE_A_TESTER = '9B71.0Z'  # exemple : choléra

ligne = df_codes[df_codes['code'] == CODE_A_TESTER]

if ligne.empty:
    print(f"{CODE_A_TESTER} : introuvable dans df_codes")
else:
    uri_test = ligne['uri'].values[0]
    axes = get_postcoord(uri_test)

    print(f"Code testé : {CODE_A_TESTER}")
    print(f"Nombre d'axes : {len(axes)}")

    for axe in axes:
        valeurs = get_valeurs_axe(axe['scale_entities'])
        print(f"  Axe '{axe['axe_nom']}' ({axe['allow_multiple']}) -> {len(valeurs)} codes, ex: {valeurs[:3]}")

Code testé : 9B71.0Z
Nombre d'axes : 3
  Axe 'laterality' (NotAllowed) -> 4 codes, ex: ['XK9J', 'XK8G', 'XK9K']
  Axe 'hasSeverity' (NotAllowed) -> 3 codes, ex: ['XS5W', 'XS0T', 'XS25']
  Axe 'hasCausingCondition' (AllowAlways) -> 14 codes, ex: ['5A10', '5A11', '5A12']


## Structures de stockage et sauvegarde de sécurité

- `chapitre_couples` : un dictionnaire `{nom_chapitre: set de couples (code_base, code_extension)}`.
  Un `set` par chapitre élimine automatiquement les doublons.
- `codes_traites` : un `set` des codes de base déjà traités, pour pouvoir
  reprendre le travail sans tout refaire si Colab coupe en cours de route.
- `sauvegarder_checkpoint()` / `charger_checkpoint()` : écrivent/lisent ces
  deux structures sur le Drive (format `pickle`, qui gère nativement les `set`).

In [ ]:
import pickle
import os

PATH_CHECKPOINT = "/content/drive/MyDrive/Colab_Notebooks/Serenic_M/api/comptage_cim11/checkpoint_couples.pkl"

def charger_checkpoint():
    """Recharge le travail déjà fait, s'il existe. Sinon, démarre à vide."""
    if os.path.exists(PATH_CHECKPOINT):
        with open(PATH_CHECKPOINT, 'rb') as f:
            data = pickle.load(f)
        print(f"Checkpoint rechargé : {len(data['codes_traites'])} codes déjà traités.")
        return data['chapitre_couples'], data['codes_traites']
    print("Aucun checkpoint trouvé, on démarre à vide.")
    return {}, set()

def sauvegarder_checkpoint(chapitre_couples, codes_traites):
    """Écrit l'état actuel sur le Drive (écrase la sauvegarde précédente)."""
    with open(PATH_CHECKPOINT, 'wb') as f:
        pickle.dump({'chapitre_couples': chapitre_couples, 'codes_traites': codes_traites}, f)

# Chargement initial (à vide la première fois)
chapitre_couples, codes_traites = charger_checkpoint()
lock_resultats = Lock()  # protège chapitre_couples / codes_traites contre les accès simultanés

Aucun checkpoint trouvé, on démarre à vide.


## Fonction de traitement d'un code de base

Pour un code donné :
1. On récupère ses axes (`get_postcoord`).
2. Pour chaque axe, on résout les codes d'extension autorisés (`get_valeurs_axe`)
   et on forme les couples `(code_base, code_extension)`.
3. On détermine son chapitre (`trouver_chapitre`).

La fonction renvoie juste ces informations — elle ne touche pas encore aux
structures partagées (`chapitre_couples`), pour éviter les conflits entre threads.

In [ ]:
def traiter_code(code: str, uri: str):
    """
    Calcule les couples (code_base, code_extension) uniques pour un code donné,
    tous axes confondus, ainsi que son chapitre d'appartenance.
    """
    couples = set()
    axes = get_postcoord(uri)

    for axe in axes:
        valeurs = get_valeurs_axe(axe['scale_entities'])
        for v in valeurs:
            couples.add((code, v))

    chapitre = trouver_chapitre(uri) or "Chapitre inconnu"
    return code, chapitre, couples

In [ ]:
CODE_A_TESTER = '9B71.0Z'  # exemple : choléra

ligne = df_codes[df_codes['code'] == CODE_A_TESTER]

if ligne.empty:
    print(f"{CODE_A_TESTER} : introuvable dans df_codes")
else:
    uri_test = ligne['uri'].values[0]
    code, chapitre, couples = traiter_code(CODE_A_TESTER, uri_test)

    print(f"Code : {code}")
    print(f"Chapitre : {chapitre}")
    print(f"Nombre de couples uniques : {len(couples)}")
    print(f"Exemples : {list(couples)[:5]}")

Code : 9B71.0Z
Chapitre : 09 - Maladies de l'appareil visuel
Nombre de couples uniques : 21
Exemples : [('9B71.0Z', '5A13.3'), ('9B71.0Z', 'XS0T'), ('9B71.0Z', 'XK9K'), ('9B71.0Z', '5A13.0'), ('9B71.0Z', '5A13.1')]


## Calcul du nombre de combinaisons possibles pour un code

Pour chaque axe d'un code, on a le choix de l'utiliser ou pas :
- Si on **n'utilise pas** l'axe : 1 possibilité (on ne met rien).
- Si on **utilise** l'axe :
  - **NotAllowed** (n valeurs) : on choisit une seule valeur parmi n → n possibilités.
  - **AllowAlways** (n valeurs) : on choisit n'importe quel sous-ensemble non vide parmi n
    (on peut combiner plusieurs valeurs du même axe) → 2ⁿ − 1 possibilités.

Pour un axe donné, le nombre total de façons de le traiter (utilisé ou pas) est donc
`1 + choix_axe`.

Le nombre de combinaisons du code = produit de ce facteur sur tous ses axes,
moins 1 à la fin (pour retirer le seul cas où **aucun** axe n'est utilisé, c'est-à-dire
le code seul sans aucune post-coordination — on ne veut compter que les vraies combinaisons).

Un code sans aucun axe n'a aucune combinaison possible (0).

In [ ]:
def calculer_nb_combinaisons(axes: list) -> int:
    """
    Calcule le nombre total de combinaisons possibles pour un code, à partir
    de la liste de ses axes (telle que renvoyée par get_postcoord), en respectant
    les règles NotAllowed (1 valeur max) / AllowAlways (sous-ensemble non vide).
    Python gère les grands entiers nativement : pas de souci mémoire même si le
    résultat a plusieurs centaines de chiffres.
    """
    if not axes:
        return 0  # pas d'axe => pas de post-coordination possible

    produit = 1
    for axe in axes:
        valeurs = get_valeurs_axe(axe['scale_entities'])
        n = len(valeurs)

        if axe['allow_multiple'] == 'AllowAlways':
            choix_axe = 2**n - 1   # tous les sous-ensembles non vides
        else:
            choix_axe = n           # une seule valeur parmi n

        produit *= (1 + choix_axe)  # "+1" = possibilité de ne pas utiliser cet axe

    return produit - 1  # on retire le cas "aucun axe utilisé"

In [ ]:
# Change juste cette ligne pour tester un autre code
CODE_A_TESTER = '9B71.0Z'

ligne = df_codes[df_codes['code'] == CODE_A_TESTER]

if ligne.empty:
    print(f"{CODE_A_TESTER} : introuvable dans df_codes")
else:
    uri_test = ligne['uri'].values[0]
    axes = get_postcoord(uri_test)
    nb_combi = calculer_nb_combinaisons(axes)

    print(f"Code : {CODE_A_TESTER}")
    print(f"Nombre d'axes : {len(axes)}")
    print(f"Nombre de combinaisons possibles : {nb_combi:,}")

Code : 9B71.0Z
Nombre d'axes : 3
Nombre de combinaisons possibles : 327,679


## Mise à jour de `traiter_code` : ajout du nombre de combinaisons

On modifie la fonction pour qu'elle calcule, en plus des couples uniques, le
nombre de combinaisons possibles du code — en une seule passe sur ses axes
(pas besoin d'appeler `get_valeurs_axe` deux fois, le cache `axe_cache` s'en
chargerait de toute façon, mais on regroupe ici pour rester lisible).

La fonction renvoie maintenant 4 éléments : `code`, `chapitre`, `couples`, `nb_combinaisons`.

In [ ]:
def traiter_code(code: str, uri: str):
    """
    Calcule, pour un code de base donné :
    - couples : l'ensemble des couples uniques (code_base, code_extension), tous axes confondus
    - nb_combinaisons : le nombre total de combinaisons possibles entre ses axes,
      en respectant les règles NotAllowed (1 valeur parmi n) / AllowAlways (sous-ensemble non vide)
    - nb_axes : le nombre d'axes de post-coordination (utile pour la sous-tâche de vérification LLM)
    - chapitre : son chapitre CIM-11 d'appartenance

    Si un axe est trop volumineux (lève AxeTropGrosError), l'exception remonte
    à la boucle principale qui marquera le code comme "impossible".
    """
    couples = set()
    produit = 1
    axes = get_postcoord(uri)

    for axe in axes:
        # Peut lever AxeTropGrosError si l'axe est trop gros → on laisse remonter
        valeurs = get_valeurs_axe(axe['scale_entities'])
        n = len(valeurs)

        # Couples uniques : une valeur à la fois, peu importe l'axe d'origine
        for v in valeurs:
            couples.add((code, v))

        # Combinaisons : nombre de façons de traiter cet axe (l'utiliser ou pas)
        choix_axe = n
        produit *= (1 + choix_axe)  # "+1" = possibilité de ne pas utiliser cet axe

    nb_combinaisons = (produit - 1) if axes else 0  # on retire le cas "aucun axe utilisé"
    nb_axes = len(axes)
    chapitre = trouver_chapitre(uri) or "Chapitre inconnu"

    return code, chapitre, couples, nb_combinaisons, nb_axes


## Mise à jour du checkpoint : ajout du total de combinaisons

En plus de `chapitre_couples` (couples uniques par chapitre), on ajoute
`chapitre_nb_combinaisons` : un dictionnaire `{chapitre: total (grand entier)}`,
qui cumule la somme des combinaisons de chaque code traité dans ce chapitre.

L'ancien fichier de checkpoint ne contient pas encore cette clé : on utilise
`.get()` pour repartir de zéro sur cette nouvelle métrique sans perdre le
travail déjà fait sur les couples uniques.

In [ ]:
def charger_checkpoint():
    """Recharge le travail déjà fait, s'il existe. Sinon, démarre à vide."""
    if os.path.exists(PATH_CHECKPOINT):
        with open(PATH_CHECKPOINT, 'rb') as f:
            data = pickle.load(f)
        chapitre_couples = data['chapitre_couples']
        codes_traites = data['codes_traites']
        # Métriques ajoutées ultérieurement : fallback vide pour les anciens checkpoints
        chapitre_nb_combinaisons = data.get('chapitre_nb_combinaisons', {})
        code_racine_info         = data.get('code_racine_info', {})
        codes_impossibles        = data.get('codes_impossibles', {})  # {code: libelle}
        print(f"Checkpoint rechargé : {len(codes_traites)} codes traités, "
              f"{len(codes_impossibles)} codes impossibles déjà identifiés.")
        return (chapitre_couples, chapitre_nb_combinaisons, code_racine_info,
                codes_impossibles, codes_traites)
    print("Aucun checkpoint trouvé, on démarre à vide.")
    return {}, {}, {}, {}, set()


def sauvegarder_checkpoint(chapitre_couples, chapitre_nb_combinaisons,
                           code_racine_info, codes_impossibles, codes_traites):
    """Écrit l'état actuel sur le Drive (écrase la sauvegarde précédente)."""
    with open(PATH_CHECKPOINT, 'wb') as f:
        pickle.dump({
            'chapitre_couples':         chapitre_couples,
            'chapitre_nb_combinaisons': chapitre_nb_combinaisons,
            'code_racine_info':         code_racine_info,
            'codes_impossibles':        codes_impossibles,
            'codes_traites':            codes_traites,
        }, f)


# Rechargement initial (à vide la première fois, ou reprise d'un travail précédent)
(chapitre_couples, chapitre_nb_combinaisons, code_racine_info,
 codes_impossibles, codes_traites) = charger_checkpoint()
lock_resultats = Lock()


Aucun checkpoint trouvé, on démarre à vide.


In [ ]:
# Vérification : on doit retrouver les mêmes résultats que les tests précédents
# (21 couples uniques, 327 679 combinaisons, 3 axes)
CODE_A_TESTER = '9B71.0Z'

ligne = df_codes[df_codes['code'] == CODE_A_TESTER]
uri_test = ligne['uri'].values[0]

code, chapitre, couples, nb_combinaisons, nb_axes = traiter_code(CODE_A_TESTER, uri_test)

print(f"Code : {code}")
print(f"Chapitre : {chapitre}")
print(f"Nombre d'axes : {nb_axes}")
print(f"Nombre de couples uniques : {len(couples)}")
print(f"Nombre de combinaisons possibles : {nb_combinaisons:,}")


Code : 9B71.0Z
Chapitre : 09 - Maladies de l'appareil visuel
Nombre d'axes : 3
Nombre de couples uniques : 21
Nombre de combinaisons possibles : 299


## Traitement de tous les codes de base

On parcourt les 34 663 codes avec `ThreadPoolExecutor` (10 threads en parallèle,
`MAX_WORKERS`). Pour chaque code traité :
1. On calcule ses couples uniques et son nombre de combinaisons (`traiter_code`).
2. On les ajoute au tiroir du bon chapitre, protégé par un verrou (`lock_resultats`)
   pour éviter que deux threads écrivent en même temps.
3. On marque le code comme traité (`codes_traites`).

**Reprise automatique** : les codes déjà présents dans `codes_traites` (rechargés
depuis le checkpoint) sont ignorés — si la session coupe, on relance la même
cellule et elle continue là où elle s'était arrêtée.

**Sauvegarde régulière** : toutes les 500 codes nouvellement traités, on
sauvegarde sur le Drive (`sauvegarder_checkpoint`).

**Suivi de progression** : un message tous les 500 codes indique l'avancement
et le temps écoulé, sans rien afficher de plus pour ne pas saturer la sortie.

In [ ]:
# ── Identification des codes fils via le cache ────────────────────────
# Un code est "fils" (feuille dans la CIM-11) s'il n'a pas d'enfants (child vide)

import pickle
PATH_CACHE = '/content/drive/MyDrive/Colab_Notebooks/Serenic_M/api/postcoord_reference/cache_get_url.pkl'

with open(PATH_CACHE, 'rb') as f:
    cache_url = pickle.load(f)
print(f"Cache chargé : {len(cache_url)} URI")

codes_fils = set()
for _, row in df_codes.iterrows():
    uri = row['uri']
    # Le cache utilise https, l'API renvoie parfois http
    data = cache_url.get(uri) or cache_url.get(uri.replace('http://', 'https://'))
    if data and not data.get('child', []):
        codes_fils.add(row['code'])

print(f"Codes fils identifiés : {len(codes_fils)} / {len(df_codes)}")

Cache chargé : 68855 URI
Codes fils identifiés : 30298 / 34663


In [ ]:
import time as _time
from tqdm.auto import tqdm

# Codes restant à traiter (on exclut ceux déjà traités ET ceux déjà impossibles)
codes_deja_vus = codes_traites | set(codes_impossibles.keys())
codes_a_traiter = df_codes[
    df_codes['code'].isin(codes_fils) & ~df_codes['code'].isin(codes_deja_vus)
]

print(f"Codes déjà traités       : {len(codes_traites)}")
print(f"Codes déjà impossibles   : {len(codes_impossibles)}")
print(f"Codes restants           : {len(codes_a_traiter)}")

CHECKPOINT_TOUS_LES = 500  # fréquence de sauvegarde (en nb de codes traités)
debut = _time.time()
compteur_depuis_save = 0


Codes déjà traités       : 0
Codes déjà impossibles   : 0
Codes restants           : 30298


In [ ]:
print(f"Codes fils au total      : {len(codes_fils)}")

Codes fils au total      : 30298


In [ ]:
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # On soumet tous les codes restants au pool de threads
    futures = {
        executor.submit(traiter_code, row['code'], row['uri']): row['code']
        for _, row in codes_a_traiter.iterrows()
    }

    # Barre de progression : codes faits / total, vitesse, ETA
    pbar = tqdm(as_completed(futures), total=len(futures), desc="Traitement", unit="code")

    for future in pbar:
        code_traite = futures[future]
        try:
            code, chapitre, couples, nb_combinaisons, nb_axes = future.result()
        except AxeTropGrosError as e:
            # Le code a au moins un axe trop volumineux → on le marque "impossible"
            # On stocke True comme valeur (le libellé sera récupéré au moment de l'export)
            with lock_resultats:
                codes_impossibles[code_traite] = True
                compteur_depuis_save += 1
            tqdm.write(f"⛔ {code_traite} : {e}")
            continue
        except Exception as e:
            tqdm.write(f"❌ Erreur sur {code_traite} : {e}")
            continue

        with lock_resultats:
            # Ajout des couples au tiroir du chapitre (set -> dédoublonnage automatique)
            chapitre_couples.setdefault(chapitre, set()).update(couples)
            # Ajout du nombre de combinaisons (grand entier, addition simple)
            chapitre_nb_combinaisons[chapitre] = chapitre_nb_combinaisons.get(chapitre, 0) + nb_combinaisons
            # Stockage de l'info détaillée par code racine (pour export CSV)
            code_racine_info[code] = {
                'chapitre':        chapitre,
                'nb_axes':         nb_axes,
                'nb_couples':      len(couples),
                'nb_combinaisons': nb_combinaisons,
            }
            codes_traites.add(code)
            compteur_depuis_save += 1

        if compteur_depuis_save >= CHECKPOINT_TOUS_LES:
            with lock_resultats:
                sauvegarder_checkpoint(chapitre_couples, chapitre_nb_combinaisons,
                                       code_racine_info, codes_impossibles, codes_traites)
                compteur_depuis_save = 0
            tqdm.write(f"[Checkpoint] {len(codes_traites)} traités, "
                       f"{len(codes_impossibles)} impossibles — "
                       f"{(_time.time() - debut)/60:.1f} min écoulées")

# Sauvegarde finale (au cas où le dernier lot n'a pas atteint le seuil de checkpoint)
sauvegarder_checkpoint(chapitre_couples, chapitre_nb_combinaisons,
                       code_racine_info, codes_impossibles, codes_traites)

print(f"\nTraitement terminé.")
print(f"  Codes traités avec succès : {len(codes_traites)}")
print(f"  Codes impossibles         : {len(codes_impossibles)}")

Traitement:   0%|          | 0/30298 [00:00<?, ?code/s]

⛔ 2A0Z : Axe trop long à résoudre : >300s écoulés (945 valeurs récupérées avant abandon)
⛔ 2E6Y : Axe trop long à résoudre : >300s écoulés (1024 valeurs récupérées avant abandon)
⛔ 2E6Z : Axe trop long à résoudre : >300s écoulés (1024 valeurs récupérées avant abandon)
⛔ 2F9C : Axe trop long à résoudre : >300s écoulés (1025 valeurs récupérées avant abandon)
⛔ 2F9Y : Axe trop long à résoudre : >300s écoulés (1024 valeurs récupérées avant abandon)
⛔ 2F9Z : Axe trop long à résoudre : >300s écoulés (1023 valeurs récupérées avant abandon)
⛔ 2F7C : Axe trop long à résoudre : >300s écoulés (1026 valeurs récupérées avant abandon)
⛔ 2F7Y : Axe trop long à résoudre : >300s écoulés (1023 valeurs récupérées avant abandon)
⛔ 2F7Z : Axe trop long à résoudre : >300s écoulés (1024 valeurs récupérées avant abandon)
⛔ 1B50 : Axe trop long à résoudre : >300s écoulés (935 valeurs récupérées avant abandon)
⛔ 1B97 : Axe trop long à résoudre : >300s écoulés (992 valeurs récupérées avant abandon)
⛔ 1C80 : Axe 

## Export du fichier "nombre de post-coordonnés par code racine"

À partir de la structure `code_racine_info` peuplée pendant le traitement, on génère un CSV récapitulatif qui donne, pour chaque code racine CIM-11 :
- le **chapitre** d'appartenance
- le **nombre d'axes** de post-coordination
- le **nombre de couples uniques** (code_base, code_extension)
- le **nombre de combinaisons possibles** (produit cartésien des choix sur tous les axes)

Ce fichier sera utilisé en entrée de la sous-tâche de **vérification LLM** : pour chaque code racine, on demandera à Mistral de générer jusqu'à 200 codes post-coordonnés réalistes à partir des axes du code.


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Export CSV : nombre de post-coordonnés par code racine
# ──────────────────────────────────────────────────────────────────────
import pandas as pd

# Construction du DataFrame à partir du dict code_racine_info
df_postcoord_par_code = pd.DataFrame([
    {
        'code':             code,
        'chapitre':         info['chapitre'],
        'nb_axes':          info['nb_axes'],
        'nb_couples':       info['nb_couples'],
        'nb_combinaisons':  info['nb_combinaisons'],
    }
    for code, info in code_racine_info.items()
], dtype=object)

# Tri par nombre de combinaisons décroissant (les plus complexes en haut)
df_postcoord_par_code = df_postcoord_par_code.sort_values('nb_combinaisons', ascending=False)

# Chemin de sortie sur le Drive
PATH_OUTPUT_CSV = "/content/drive/MyDrive/Colab_Notebooks/Serenic_M/api/comptage_cim11/nb_postcoord_par_code_racine.csv"

import sys, math
sys.set_int_max_str_digits(0)

df_postcoord_par_code.to_csv(PATH_OUTPUT_CSV, index=False, encoding='utf-8-sig')


def format_nb(n):
    n = int(n)
    if n < 10**6:
        return f"{n:,}"
    total_str = str(n)
    n_digits = len(total_str)
    log_n = math.log10(n)
    mantisse = total_str[0] + '.' + total_str[1:6]
    return f"{mantisse} × 10^{n_digits-1} (~10^{log_n:.1f})"


print(f"Fichier exporté : {PATH_OUTPUT_CSV}")
print(f"Nombre de codes : {len(df_postcoord_par_code)}")
print()

# Aperçu avec formatage scientifique pour les grands nombres
apercu = df_postcoord_par_code.head(10).copy()
apercu['nb_combinaisons'] = apercu['nb_combinaisons'].apply(format_nb)
print("Aperçu (10 codes avec le plus de combinaisons) :")
print(apercu.to_string(index=False))
print()

print("Statistiques globales :")
print(f"  Codes sans post-coordination : {(df_postcoord_par_code['nb_combinaisons'] == 0).sum()}")
print(f"  Codes avec >200 combinaisons : {sum(1 for x in df_postcoord_par_code['nb_combinaisons'] if int(x) > 200)}")

max_n = max(int(x) for x in df_postcoord_par_code['nb_combinaisons'])
total = sum(int(x) for x in df_postcoord_par_code['nb_combinaisons'])
print(f"  Maximum : {format_nb(max_n)}")
print(f"  Total   : {format_nb(total)}")

Fichier exporté : /content/drive/MyDrive/Colab_Notebooks/Serenic_M/api/comptage_cim11/nb_postcoord_par_code_racine.csv
Nombre de codes : 29177

Aperçu (10 codes avec le plus de combinaisons) :
   code                                                                                     chapitre nb_axes nb_couples          nb_combinaisons
   PA1E                                            23 - Causes externes de morbidité ou de mortalité       6        463 1.43366 × 10^9 (~10^9.2)
   ND9Y 22 - Lésions traumatiques, intoxications ou certaines autres conséquences de causes externes       6       1350 1.27250 × 10^9 (~10^9.1)
   ND9Z 22 - Lésions traumatiques, intoxications ou certaines autres conséquences de causes externes       6       1350 1.27250 × 10^9 (~10^9.1)
NA07.61 22 - Lésions traumatiques, intoxications ou certaines autres conséquences de causes externes       7       1017 1.03002 × 10^9 (~10^9.0)
NA07.60 22 - Lésions traumatiques, intoxications ou certaines autres conséquences 

## Export du fichier des codes impossibles

Certains codes racines ont au moins un axe de post-coordination "trop volumineux" pour être traité automatiquement (typiquement `associatedWith` qui peut référencer toutes les maladies, ou `histopathology` avec des milliers de types histologiques).

Ces codes sont identifiés par la levée d'`AxeTropGrosError` pendant le traitement et stockés dans `codes_impossibles`. On les exporte dans un fichier à part pour pouvoir les **traiter manuellement** ou les **exclure** de la suite du pipeline.


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Export CSV : codes racines impossibles à post-coordonner automatiquement
# (au moins un axe trop volumineux pour être proposé à un LLM)
# ──────────────────────────────────────────────────────────────────────
codes_imp_list = list(codes_impossibles.keys())

# On récupère le libellé depuis df_syn (qui contient les codes + libellés en colonne 'texte')
# On filtre sur type='title' pour avoir le libellé officiel (pas les synonymes)
df_impossibles = (
    df_syn[
        (df_syn['code'].isin(codes_imp_list))
        & (df_syn['type'] == 'title')
    ][['code', 'texte']]
    .drop_duplicates(subset='code')
    .rename(columns={'texte': 'libelle'})
    .sort_values('code')
)

PATH_OUTPUT_IMPOSSIBLES = "/content/drive/MyDrive/Colab_Notebooks/Serenic_M/api/comptage_cim11/codes_impossibles.csv"
df_impossibles.to_csv(PATH_OUTPUT_IMPOSSIBLES, index=False, encoding='utf-8-sig')

print(f"Fichier exporté : {PATH_OUTPUT_IMPOSSIBLES}")
print(f"Nombre de codes impossibles : {len(df_impossibles)}")
print()
print("Aperçu (20 premiers codes) :")
print(df_impossibles.head(20).to_string(index=False))

# Vérification : codes impossibles sans libellé trouvé dans df_syn
codes_sans_libelle = set(codes_imp_list) - set(df_impossibles['code'])
if codes_sans_libelle:
    print(f"\n⚠️  {len(codes_sans_libelle)} codes sans libellé trouvé dans df_syn :")
    for c in list(codes_sans_libelle)[:10]:
        print(f"    {c}")

Fichier exporté : /content/drive/MyDrive/Colab_Notebooks/Serenic_M/api/comptage_cim11/codes_impossibles.csv
Nombre de codes impossibles : 1121

Aperçu (20 premiers codes) :
   code                                                       libelle
1A36.1Y         Autres infections extra-intestinales dues à Entamoeba
 1A72.0         Infection gonococcique du système musculosquelettique
   1B50                                                    Scarlatine
 1B72.2                          Impétiginisation d'autres dermatoses
   1B97                                                       Anthrax
   1C2Y                        Autres autres maladies due à Chlamydia
   1C2Z               Autres maladies due à Chlamydia, sans précision
   1C80                      Encéphalite virale, non classée ailleurs
 1D00.Y          Autres encéphalite infectieuse, non classée ailleurs
 1D00.Z Encéphalite infectieuse, non classée ailleurs, sans précision
1D01.0Y                      Autres méningites bactérienn

In [ ]:
import pickle

with open('/content/drive/MyDrive/Colab_Notebooks/Serenic_M/api/postcoord_reference/cache_get_url.pkl', 'rb') as f:
    cache_url = pickle.load(f)

# On cherche 3 exemples d'arborescence : nœud → ses enfants
def afficher_noeud(uri, indent=0):
    data = cache_url.get(uri)
    if not data:
        return
    code = data.get('code', '(sans code)')
    titre = data.get('title', {}).get('@value', '')
    enfants = data.get('child', [])
    marqueur = "FEUILLE" if not enfants else f"{len(enfants)} enfants"
    print(f"{'  ' * indent}[{marqueur}] {code} — {titre}")
    return enfants


# Exemple 1 : partons d'un nœud connu avec enfants
print("=== Exemple : hiérarchie autour du code 2A00 ===")
uri_2A00 = 'https://id.who.int/icd/release/11/2024-01/mms/1435254666'  # exemple, à adapter si besoin

# Chercher un URI dans le cache qui correspond au code 2A00
for uri, data in cache_url.items():
    if data and data.get('code') == '2A00':
        print(f"\nURI : {uri}")
        enfants = afficher_noeud(uri, 0)
        # Afficher aussi les 3 premiers enfants
        for child_uri in enfants[:3]:
            afficher_noeud(child_uri.replace('http://', 'https://'), 1)
        break

# Exemple 2 : chercher une feuille pure (sans enfants)
print("\n=== Exemple de feuilles pures (extension X) ===")
n = 0
for uri, data in cache_url.items():
    if data is None:
        continue
    code = data.get('code', '')
    if code.startswith('X') and not data.get('child', []):
        afficher_noeud(uri, 0)
        n += 1
        if n >= 3:
            break

# Exemple 3 : regroupement pur (sans code, avec enfants)
print("\n=== Exemple de regroupements purs (sans code, avec enfants) ===")
n = 0
for uri, data in cache_url.items():
    if data is None:
        continue
    if not data.get('code') and data.get('child'):
        titre = data.get('title', {}).get('@value', '')
        n_enfants = len(data.get('child', []))
        print(f"  Regroupement : '{titre}' → {n_enfants} enfants")
        n += 1
        if n >= 3:
            break

=== Exemple : hiérarchie autour du code 2A00 ===

URI : http://id.who.int/icd/release/11/2024-01/mms/1719389232
[6 enfants] 2A00 — Tumeurs primitives du cerveau
  [3 enfants] 2A00.0 — Gliomes du cerveau
  [4 enfants] 2A00.1 — Tumeurs embryonnaires du cerveau
  [5 enfants] 2A00.2 — Tumeurs du tissu neuroépithélial du cerveau

=== Exemple de feuilles pures (extension X) ===
[FEUILLE] XT9T — Lié au vieillissement
[FEUILLE] XT13 — Période gériatrique tardive
[FEUILLE] XT19 — Période gériatrique précoce

=== Exemple de regroupements purs (sans code, avec enfants) ===
  Regroupement : 'Effets autres ou non précisés de causes externes' → 13 enfants
  Regroupement : 'Topographie de surface' → 3 enfants
  Regroupement : 'Causes de lésions ou de préjudices liés aux soins' → 8 enfants
